In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
Tesla T4


In [2]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

In [3]:
from datasets import load_dataset
import pandas as pd

raw = load_dataset("winvoker/turkish-sentiment-analysis-dataset")

train_df = raw["train"].to_pandas()
test_df = raw["test"].to_pandas()

train_df = train_df[train_df["label"] != "Notr"].copy()
test_df = test_df[test_df["label"] != "Notr"].copy()

train_df["label_num"] = (train_df["label"] == "Positive").astype(int)
test_df["label_num"] = (test_df["label"] == "Positive").astype(int)

print(train_df.shape, test_df.shape)

(286854, 4) (31873, 4)


In [4]:
from sklearn.model_selection import train_test_split

# Her sınıftan orantılı şekilde 25.000 satırlık bir örneklem alalım
train_sample, _ = train_test_split(
    train_df,
    train_size=25000,
    stratify=train_df["label_num"],
    random_state=42
)

test_sample, _ = train_test_split(
    test_df,
    train_size=3000,
    stratify=test_df["label_num"],
    random_state=42
)

print(train_sample.shape, test_sample.shape)
print(train_sample["label_num"].value_counts())

(25000, 4) (3000, 4)
label_num
1    20564
0     4436
Name: count, dtype: int64


In [5]:
from transformers import AutoTokenizer

model_name = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Örnek bir cümle üzerinde deneyelim
ornek = "Bu ürün gerçekten harika, kesinlikle tavsiye ederim!"
print(tokenizer(ornek))
print(tokenizer.tokenize(ornek))

{'input_ids': [2, 2123, 2782, 4036, 5412, 16, 5428, 5668, 5002, 5, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
['Bu', 'ürün', 'gerçekten', 'harika', ',', 'kesinlikle', 'tavsiye', 'ederim', '!']


In [6]:
# Önce cümle uzunluklarına bir bakalım, doğru max_length'i seçmek için
lengths = train_sample["text"].apply(lambda x: len(tokenizer.tokenize(str(x))))
print(lengths.describe())

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (621 > 512). Running this sequence through the model will result in indexing errors


count    25000.000000
mean        38.654480
std         48.564392
min          1.000000
25%         14.000000
50%         28.000000
75%         46.000000
max       3122.000000
Name: text, dtype: float64


In [7]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_sample[["text", "label_num"]].reset_index(drop=True))
test_ds = Dataset.from_pandas(test_sample[["text", "label_num"]].reset_index(drop=True))

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column("label_num", "labels")
test_ds = test_ds.rename_column("label_num", "labels")

print(train_ds)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25000
})


In [8]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
import numpy as np
from sklearn.metrics import classification_report, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, preds, average="macro")
    return {"macro_f1": macro_f1}

In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert_output",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    learning_rate=2e-5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,0.193677,0.151895,0.901947
2,0.108364,0.175902,0.921308


TrainOutput(global_step=3126, training_loss=0.1516888930259114, metrics={'train_runtime': 1126.835, 'train_samples_per_second': 44.372, 'train_steps_per_second': 2.774, 'total_flos': 3288888192000000.0, 'train_loss': 0.1516888930259114, 'epoch': 2.0})

In [12]:
predictions = trainer.predict(test_ds)
y_pred_bert = np.argmax(predictions.predictions, axis=1)
y_true_bert = predictions.label_ids

print(classification_report(y_true_bert, y_pred_bert, target_names=["Negative", "Positive"]))

              precision    recall  f1-score   support

    Negative       0.89      0.85      0.87       532
    Positive       0.97      0.98      0.97      2468

    accuracy                           0.95      3000
   macro avg       0.93      0.91      0.92      3000
weighted avg       0.95      0.95      0.95      3000



In [13]:
trainer.save_model("./bert_finetuned_model")
tokenizer.save_pretrained("./bert_finetuned_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bert_finetuned_model/tokenizer_config.json',
 './bert_finetuned_model/tokenizer.json')

In [14]:
!zip -r bert_finetuned_model.zip bert_finetuned_model

  adding: bert_finetuned_model/ (stored 0%)
  adding: bert_finetuned_model/training_args.bin (deflated 54%)
  adding: bert_finetuned_model/config.json (deflated 52%)
  adding: bert_finetuned_model/tokenizer.json (deflated 70%)
  adding: bert_finetuned_model/model.safetensors (deflated 7%)
  adding: bert_finetuned_model/tokenizer_config.json (deflated 46%)
